In [1]:
from pathlib import Path
import os
# The project folder is the folder that contains data/, notebooks/, results/ ...
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
PROJECT_DIR = Path(os.environ.get('PROJECT_DIR', PROJECT_DIR))
print('Project folder:', PROJECT_DIR)
assert (PROJECT_DIR / 'data').exists(), 'Project folder not found - open this notebook from the notebooks folder of the project (see your Steps file)'

Project folder: C:\Users\Reach\Documents\game-chat-toxicity-ml


In [2]:
import re, html, json
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
import matplotlib.pyplot as plt

USE_SAMPLE = False
TRAINABLE  = False
MAX_EPOCHS = 100
SEED = 42
tf.keras.utils.set_random_seed(SEED)

LAB_DIR, EMB_DIR, RES_DIR, MODEL_DIR = PROJECT_DIR / 'data' / 'labelled', PROJECT_DIR / 'models' / 'emb', PROJECT_DIR / 'results', PROJECT_DIR / 'models'
(RES_DIR / 'preds').mkdir(parents=True, exist_ok=True); (RES_DIR / 'figures').mkdir(parents=True, exist_ok=True)
prefix = 'sample_' if USE_SAMPLE else ''
train = pd.read_csv(LAB_DIR / f'{prefix}train.csv'); test = pd.read_csv(LAB_DIR / f'{prefix}test.csv')
meta = json.load(open(EMB_DIR / 'vocab.json')); vocab, MAX_LEN = meta['vocab'], meta['max_len']

def clean_text(s):
    s = html.unescape(str(s)).lower()
    s = re.sub(r"https?://\S+", " ", s)
    s = re.sub(r"[\u2019'`]", "", s)
    s = re.sub(r"(.)\1{2,}", r"\1\1", s)
    s = re.sub(r"[^a-z0-9?! ]+", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def encode(texts):
    X = np.zeros((len(texts), MAX_LEN), dtype='int32')
    for i, s in enumerate(texts):
        ids = [vocab.get(w, 1) for w in re.findall(r"[a-z0-9]+|[?!]", clean_text(s))][:MAX_LEN]
        X[i, :len(ids)] = ids
    return X

X_all, y_all = encode(train.text), train.label.values
X_test, y_test = encode(test.text), test.label.values
X_tr, X_val, y_tr, y_val = train_test_split(X_all, y_all, test_size=0.2, stratify=y_all, random_state=SEED)
print('train/validation/test sizes:', len(X_tr), len(X_val), len(X_test))


train/validation/test sizes: 1266 317 396


In [3]:
EMBEDDINGS = {'word2vec 50-dim': 'w2v_50', 'word2vec 100-dim': 'w2v_100', 'GloVe 50-dim': 'glove_50', 'GloVe 100-dim': 'glove_100'}
OUTPUTS = ['softmax', 'sigmoid']
onehot = lambda y: tf.keras.utils.to_categorical(y, 3)

def build_lstm(emb, output):
    inp = layers.Input(shape=(MAX_LEN,))
    x = layers.Embedding(emb.shape[0], emb.shape[1], weights=[emb], trainable=TRAINABLE, mask_zero=True)(inp)
    x = layers.LSTM(64)(x)
    x = layers.Dropout(0.2)(x)
    out = layers.Dense(3, activation=output)(x)
    m = models.Model(inp, out)
    loss = 'sparse_categorical_crossentropy' if output == 'softmax' else 'binary_crossentropy'
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss=loss, metrics=['accuracy'])
    return m

def targets(y, output):
    return y if output == 'softmax' else onehot(y)

results, histories, best_val, best_info = [], {}, -1, None
for emb_name, emb_file in EMBEDDINGS.items():
    emb = np.load(EMB_DIR / f'{emb_file}.npy')
    for output in OUTPUTS:
        tf.keras.utils.set_random_seed(SEED)
        model = build_lstm(emb, output)
        stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)
        h = model.fit(X_tr, targets(y_tr, output), validation_data=(X_val, targets(y_val, output)),
                      epochs=MAX_EPOCHS, batch_size=64, callbacks=[stop], verbose=0)
        pred = model.predict(X_test, verbose=0).argmax(1)
        val_pred = model.predict(X_val, verbose=0).argmax(1)
        val_acc = accuracy_score(y_val, val_pred)
        f1s = f1_score(y_test, pred, labels=[0, 1, 2], average=None, zero_division=0)
        name = f'LSTM ({output} output; {emb_name})'
        row = {'model': name, 'family': 'Neural net', 'features': emb_name, 'task': '3-class',
               'accuracy': round(accuracy_score(y_test, pred), 4), 'macro_f1': round(float(f1s.mean()), 4),
               'f1_class0': round(float(f1s[0]), 4), 'f1_class1': round(float(f1s[1]), 4), 'f1_class2': round(float(f1s[2]), 4),
               'val_accuracy': round(val_acc, 4), 'epochs': len(h.history['loss'])}
        results.append(row); histories[name] = h.history
        slug = re.sub(r'[^a-z0-9]+', '_', name.lower()).strip('_')
        pd.DataFrame({'id': test.id.values, 'y_true': y_test, 'y_pred': pred}).to_csv(RES_DIR / 'preds' / f'{slug}.csv', index=False)
        print(f"{name:52s} acc={row['accuracy']:.4f} macroF1={row['macro_f1']:.4f} val_acc={val_acc:.4f} epochs={row['epochs']}")
        if val_acc > best_val:                                   # choose the demo model on the VALIDATION set, never on the test set
            best_val, best_info = val_acc, (name, emb_file, output)
            if not USE_SAMPLE: model.save(MODEL_DIR / 'lstm_best.keras')
print('\nBest LSTM on validation data:', best_info)

LSTM (softmax output; word2vec 50-dim)               acc=0.8308 macroF1=0.6531 val_acc=0.8170 epochs=16
LSTM (sigmoid output; word2vec 50-dim)               acc=0.8157 macroF1=0.6132 val_acc=0.8202 epochs=19
LSTM (softmax output; word2vec 100-dim)              acc=0.8434 macroF1=0.7226 val_acc=0.8044 epochs=16
LSTM (sigmoid output; word2vec 100-dim)              acc=0.8409 macroF1=0.7124 val_acc=0.8107 epochs=23
LSTM (softmax output; GloVe 50-dim)                  acc=0.8384 macroF1=0.7123 val_acc=0.8675 epochs=35
LSTM (sigmoid output; GloVe 50-dim)                  acc=0.8232 macroF1=0.6705 val_acc=0.8864 epochs=31
LSTM (softmax output; GloVe 100-dim)                 acc=0.8359 macroF1=0.6831 val_acc=0.8738 epochs=25
LSTM (sigmoid output; GloVe 100-dim)                 acc=0.8283 macroF1=0.7061 val_acc=0.8738 epochs=30

Best LSTM on validation data: ('LSTM (sigmoid output; GloVe 50-dim)', 'glove_50', 'sigmoid')
